# Hangman â€” Brand & Buzzword Hackathon

A character-level Transformer that plays Hangman, trained only on the provided
`train.txt`.

## The problem, as the data actually presents it

The corpus is an **English dictionary word list** â€” lowercase `a-z`, no digits or
punctuation. Two properties shape every decision below:

| | train.txt | test.txt |
|---|---|---|
| words | 225,300 | 250,000 |
| mean length | 9.35 | 9.43 |
| charset | `a-z` | `a-z` |

**The two sets share zero words.** Memorisation is impossible by construction, so
only generalisation over spelling patterns can score. That is not a detail â€” it
is the whole problem.

## Why a learned model, measured rather than asserted

Reference policies on held-out training words:

| policy | win rate |
|---|---|
| static letter frequency (= `sample_submission.csv`) | 12.70% |
| pattern matching over the training vocabulary | 15.65% |

Filtering the training vocabulary to words consistent with the board â€” the
standard statistical approach â€” barely beats a fixed letter order, **because the
vocabularies are disjoint**. It can only recognise spellings it has already seen.
A model that learns orthography rather than a word list is not a stylistic
preference here; it is the only thing that generalises.

## Structure

The notebook inlines a tested Python package. Each section below is one module,
in dependency order, so the code that runs here is the code the test suite
covers.


## 1. Game engine

Exact competition semantics, and the boundary that makes cheating impossible.

`test.txt` ships with its answers, so a solver *could* score 100% by reading them -- which the rules treat as disqualifying. Rather than rely on discipline, policies are handed an `Observation` that contains the board, the guess history and the strike count, and **never the secret word**. The model is structurally incapable of seeing the answer.

The engine also pins the scoring rules that are easy to get wrong: a repeated guess, a non-letter, or a letter not in the word each cost exactly one strike, and a game stops the instant it is won or hits six.


In [ ]:
"""Hangman game engine implementing exact competition semantics.

Rules (from the competition specification):

* A secret word is hidden as underscores. Non-letter characters (spaces, digits,
  punctuation) are revealed from the start -- only ``a-z`` is ever guessed.
* A guess that reveals at least one new letter is a hit; every other guess is
  exactly one strike. That includes wrong letters, repeated/duplicate guesses,
  spaces, digits and invalid symbols.
* A game ends the instant the word is fully revealed (win) or the strike count
  reaches ``MAX_WRONG_GUESSES`` (loss). Any guesses after that point are ignored.

The engine is deliberately policy-agnostic: it knows how to run a game but not
how to choose a letter. Policies implement :class:`GuessPolicy`.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Iterable, Protocol, Sequence

ALPHABET: str = "abcdefghijklmnopqrstuvwxyz"
LETTER_TO_INDEX: dict[str, int] = {letter: i for i, letter in enumerate(ALPHABET)}
MAX_WRONG_GUESSES: int = 6
MASK_CHAR: str = "_"


def is_guessable(char: str) -> bool:
    """Return whether ``char`` is a letter the player is allowed to guess."""
    return char in LETTER_TO_INDEX


@dataclass(frozen=True)
class Observation:
    """Everything a policy is allowed to know about a game in progress.

    This type deliberately excludes the secret word. The public ``test.txt``
    contains the answers, so a policy that could reach the word would be able to
    cheat -- accidentally or otherwise -- and the competition treats lookup
    leaks as disqualifying. Making the observation the only channel between the
    engine and a policy turns that from a discipline problem into a structural
    guarantee.

    Attributes:
        board: Per-character view, with ``MASK_CHAR`` for hidden letters.
        guesses: Every character submitted so far, in chronological order.
        wrong_guesses: Strikes accumulated so far.
    """

    board: str
    guesses: tuple[str, ...]
    wrong_guesses: int

    @property
    def length(self) -> int:
        return len(self.board)

    @property
    def guessed_letters(self) -> frozenset[str]:
        return frozenset(self.guesses)

    @property
    def remaining_lives(self) -> int:
        return MAX_WRONG_GUESSES - self.wrong_guesses


@dataclass
class GameState:
    """Mutable state of a single Hangman game.

    Attributes:
        word: The secret word, lowercased. Never exposed to policies.
        board: Per-character view. Hidden letters are ``MASK_CHAR``; revealed
            letters and non-letter characters show their true character.
        guesses: Every character submitted, in chronological order.
        wrong_guesses: Number of strikes accumulated so far.
    """

    word: str
    board: list[str] = field(init=False)
    guesses: list[str] = field(default_factory=list)
    wrong_guesses: int = 0

    def __post_init__(self) -> None:
        # Non-letters are visible from the start; letters begin masked.
        self.board = [MASK_CHAR if is_guessable(c) else c for c in self.word]

    @property
    def masked_word(self) -> str:
        """The board as a string, e.g. ``"_p p_ e"``."""
        return "".join(self.board)

    @property
    def guessed_letters(self) -> set[str]:
        """The set of characters already submitted."""
        return set(self.guesses)

    @property
    def is_solved(self) -> bool:
        return MASK_CHAR not in self.board

    @property
    def is_out_of_lives(self) -> bool:
        return self.wrong_guesses >= MAX_WRONG_GUESSES

    @property
    def is_over(self) -> bool:
        return self.is_solved or self.is_out_of_lives

    @property
    def guess_string(self) -> str:
        """The chronological guess sequence, as required by the submission CSV."""
        return "".join(self.guesses)

    @property
    def observation(self) -> Observation:
        """The word-free view handed to policies."""
        return Observation(
            board=self.masked_word,
            guesses=tuple(self.guesses),
            wrong_guesses=self.wrong_guesses,
        )

    def apply_guess(self, guess: str) -> bool:
        """Apply one guess and return whether it revealed a new letter.

        A guess is a hit only if it uncovers at least one previously hidden
        position. Everything else -- a letter not in the word, a repeat of an
        earlier guess, or a non-letter character -- costs exactly one strike.

        Raises:
            RuntimeError: If the game has already terminated.
            ValueError: If ``guess`` is not a single character.
        """
        if self.is_over:
            raise RuntimeError("Cannot guess: this game has already terminated.")
        if len(guess) != 1:
            raise ValueError(f"A guess must be exactly one character, got {guess!r}.")

        revealed_any = False
        if is_guessable(guess) and guess not in self.guessed_letters:
            for i, char in enumerate(self.word):
                if char == guess and self.board[i] == MASK_CHAR:
                    self.board[i] = char
                    revealed_any = True

        self.guesses.append(guess)
        if not revealed_any:
            self.wrong_guesses += 1
        return revealed_any


class GuessPolicy(Protocol):
    """Chooses the next letter for each of a batch of in-progress games.

    Policies receive :class:`Observation` values -- never the secret word -- and
    receive them in a batch so that model-based implementations can run a single
    vectorised forward pass per turn instead of one per game.
    """

    def next_guesses(self, observations: Sequence[Observation]) -> Sequence[str]:
        """Return one guess character per observation, in the same order."""
        ...


@dataclass(frozen=True)
class GameResult:
    """Outcome of a finished game."""

    word: str
    guess_string: str
    solved: bool
    wrong_guesses: int


def play_games(words: Iterable[str], policy: GuessPolicy) -> list[GameResult]:
    """Play one game per word, stepping every active game in lockstep.

    All games that are still running are advanced together so that ``policy``
    sees the largest possible batch on each turn. Results are returned in the
    same order as ``words``.
    """
    states = [GameState(word=word) for word in words]
    active = [state for state in states if not state.is_over]

    while active:
        guesses = policy.next_guesses([state.observation for state in active])
        if len(guesses) != len(active):
            raise ValueError(
                f"Policy returned {len(guesses)} guesses for {len(active)} active games."
            )
        for state, guess in zip(active, guesses):
            state.apply_guess(guess)
        active = [state for state in active if not state.is_over]

    return [
        GameResult(
            word=state.word,
            guess_string=state.guess_string,
            solved=state.is_solved,
            wrong_guesses=state.wrong_guesses,
        )
        for state in states
    ]


def summarise(results: Sequence[GameResult]) -> dict[str, float]:
    """Compute the competition metrics for a set of finished games.

    Returns the primary win rate (percentage of words solved) and the
    tie-break inputs: total and mean wrong guesses.
    """
    if not results:
        return {"games": 0, "win_rate": 0.0, "total_wrong": 0, "mean_wrong": 0.0}

    solved = sum(result.solved for result in results)
    total_wrong = sum(result.wrong_guesses for result in results)
    return {
        "games": len(results),
        "win_rate": 100.0 * solved / len(results),
        "total_wrong": total_wrong,
        "mean_wrong": total_wrong / len(results),
    }


## 2. State encoding

One encoder serves both training and inference, so the model cannot see a different feature layout in the two settings.

The 26-dim guessed vector is not redundant with the board. A letter that was guessed and *missed* never appears on the board, yet knowing it is absent is some of the most valuable information the player holds. Without it the model would be blind to its own misses.


In [ ]:
"""Tensor encoding of Hangman game states.

A single encoder serves both training and inference so the model cannot see a
different feature layout in the two settings -- a common and silent source of
train/serve skew.

Each state becomes three tensors:

* ``tokens``   -- ``(batch, max_length)`` board tokens, padded.
* ``guessed``  -- ``(batch, 26)`` binary, letters already submitted.
* ``padding``  -- ``(batch, max_length)`` boolean, ``True`` at padding positions.

The ``guessed`` vector is not redundant with the board. A letter that was
guessed and missed never appears on the board, yet knowing it is absent is some
of the most valuable information the player holds -- it eliminates candidate
spellings. Without this input the model would be blind to its own misses.
"""

from __future__ import annotations

from typing import Sequence

import numpy as np
import torch


PAD_TOKEN_ID: int = 0
MASK_TOKEN_ID: int = 1
LETTER_TOKEN_OFFSET: int = 2
VOCABULARY_SIZE: int = LETTER_TOKEN_OFFSET + len(ALPHABET)

# The longest word in the corpus is 29 characters; 32 leaves headroom for a
# private evaluation set with slightly longer entries.
MAX_WORD_LENGTH: int = 32

_LETTER_TO_TOKEN = {letter: LETTER_TOKEN_OFFSET + i for i, letter in enumerate(ALPHABET)}
_LETTER_A = ord("a")


def encode_boards(
    boards: Sequence[str],
    guessed_masks: np.ndarray,
    *,
    max_length: int = MAX_WORD_LENGTH,
) -> dict[str, np.ndarray]:
    """Encode board strings and guessed-letter masks into padded arrays.

    Args:
        boards: Board strings, using ``MASK_CHAR`` for hidden letters.
        guessed_masks: ``(batch, 26)`` binary array of already-guessed letters.
        max_length: Padded sequence length.

    Returns:
        Dict of ``tokens`` (int64), ``guessed`` (float32) and ``padding`` (bool).

    Raises:
        ValueError: If a board is longer than ``max_length``, or if the mask
            shape does not match the number of boards.
    """
    batch = len(boards)
    if guessed_masks.shape != (batch, len(ALPHABET)):
        raise ValueError(
            f"guessed_masks must be ({batch}, {len(ALPHABET)}), "
            f"got {guessed_masks.shape}."
        )

    lengths = np.fromiter((len(board) for board in boards), dtype=np.int64, count=batch)
    if batch and lengths.max() > max_length:
        longest = boards[int(lengths.argmax())]
        raise ValueError(
            f"Board of length {lengths.max()} exceeds max_length={max_length}: {longest!r}"
        )

    # Scatter every board into a padded byte matrix in one shot. Encoding one
    # character at a time in Python dominated inference: this runs on every turn
    # of every game, so 250,000 test words meant tens of millions of iterations.
    occupied = np.arange(max_length)[None, :] < lengths[:, None]
    characters = np.zeros((batch, max_length), dtype=np.uint8)
    if batch:
        characters[occupied] = np.frombuffer(
            "".join(boards).encode("ascii"), dtype=np.uint8
        )

    letter_index = characters.astype(np.int16) - _LETTER_A
    is_letter = (letter_index >= 0) & (letter_index < len(ALPHABET))

    # Anything that is not a revealed a-z letter is a blank from the model's
    # point of view. That covers MASK_CHAR and also any non-letter the private
    # set might contain: those are visible from the start and carry no
    # guessable information.
    tokens = np.where(
        is_letter, letter_index.astype(np.int64) + LETTER_TOKEN_OFFSET, MASK_TOKEN_ID
    )
    tokens[~occupied] = PAD_TOKEN_ID

    return {
        "tokens": tokens,
        "guessed": guessed_masks.astype(np.float32),
        "padding": ~occupied,
    }


def encode_observations(
    observations: Sequence[Observation],
    *,
    max_length: int = MAX_WORD_LENGTH,
    device: torch.device | str = "cpu",
) -> dict[str, torch.Tensor]:
    """Encode live :class:`Observation` values into a batch of tensors."""
    guessed = np.zeros((len(observations), len(ALPHABET)), dtype=np.float32)
    for row, observation in enumerate(observations):
        # Iterate the raw guess tuple rather than the `guessed_letters` property,
        # which builds a fresh frozenset on every access -- and this runs once
        # per game per turn.
        for letter in observation.guesses:
            index = ALPHABET.find(letter)
            if index >= 0:
                guessed[row, index] = 1.0

    arrays = encode_boards(
        [observation.board for observation in observations],
        guessed,
        max_length=max_length,
    )
    return {key: torch.as_tensor(value, device=device) for key, value in arrays.items()}


## 3. Data loading and splitting

The split is random rather than alphabetical. `train.txt` is sorted, so a positional split would put entire prefix families (every `un-` word, say) on one side and measure the wrong thing.


In [ ]:
"""Loading and splitting the competition word lists.

The corpus is an English dictionary: 225,300 training words and 250,000 test
words, lowercase ``a-z`` only, with **zero overlap** between the two. Nothing can
be memorised, so every design choice here is aimed at measuring generalisation
to unseen spellings.

Model selection uses a held-out slice of ``train.txt``. The public ``test.txt``
ships with its answers, which means a local replay reproduces the public
leaderboard score exactly -- so it is reserved for final reporting and kept out
of the tuning loop, where it would otherwise invite overfitting to the public
split at the expense of the private one.
"""

from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence


DEFAULT_DATA_DIR = Path("data")
TRAIN_FILENAME = "train.txt"
TEST_FILENAME = "test.txt"


def load_words(path: str | Path) -> list[str]:
    """Read a newline-delimited word list, preserving file order.

    File order matters for ``test.txt``: ``word_id`` in the submission is the
    zero-based line index, so the list must not be reordered or deduplicated.
    """
    with open(path, encoding="utf-8") as handle:
        return [line.strip().lower() for line in handle if line.strip()]


@dataclass(frozen=True)
class WordSplit:
    """A train/validation partition of the training vocabulary."""

    train: list[str]
    validation: list[str]

    def __str__(self) -> str:
        return (
            f"WordSplit(train={len(self.train):,}, validation={len(self.validation):,})"
        )


def split_words(
    words: Sequence[str],
    *,
    validation_size: int = 20_000,
    seed: int = 20260901,
) -> WordSplit:
    """Randomly partition ``words`` into fitting and model-selection sets.

    The split is random rather than alphabetical: ``train.txt`` is sorted, so a
    positional split would put entire prefix families (every ``un-`` word, say)
    on one side and make validation measure the wrong thing.
    """
    if validation_size >= len(words):
        raise ValueError(
            f"validation_size={validation_size} must be smaller than the "
            f"{len(words)} available words."
        )

    shuffled = list(words)
    random.Random(seed).shuffle(shuffled)
    return WordSplit(
        train=shuffled[validation_size:],
        validation=shuffled[:validation_size],
    )


def describe_vocabulary(words: Sequence[str]) -> dict[str, object]:
    """Summarise a word list: size, length range, and any out-of-alphabet characters.

    The competition rules allow non-letter characters in the vocabulary even
    though the public corpus contains none. This surfaces that discrepancy
    immediately if the private evaluation set differs.
    """
    lengths = [len(word) for word in words]
    charset = set("".join(words))
    return {
        "count": len(words),
        "unique": len(set(words)),
        "min_length": min(lengths) if lengths else 0,
        "max_length": max(lengths) if lengths else 0,
        "mean_length": sum(lengths) / len(lengths) if lengths else 0.0,
        "charset_size": len(charset),
        "non_alphabet_characters": sorted(charset - set(ALPHABET)),
    }


## 4. Training-state generation

**The most important component, and the least obvious.**

The tempting approach -- reveal a random subset of a word's letters -- produces board states that never arise in play. A real game reveals letters in the order a player guessed them and carries the scars of that player's misses. A model fitted to uniform random subsets is answering a different question at inference than the one it was trained on.

So states are drawn from *simulated trajectories*: each word is played out by a stochastic frequency-ranked player and a uniformly random turn along that trajectory is sampled. Whole games are simulated in closed form with cumulative sums, which makes generating millions of states cost seconds. A quarter of games use a flat ranking, reaching unusual states a competent player rarely visits.


In [ ]:
"""Sampling training states from simulated games.

The subtle part of this problem is not the architecture, it is *which board
states the model is trained on*. The obvious approach -- reveal a random subset
of a word's letters -- produces states that never arise in play. A real game
reveals letters in the order a player guesses them, and carries the scars of
that player's misses. A model fitted to uniform random subsets is being asked a
different question at inference time than the one it was trained on.

States here are drawn from *simulated trajectories* instead. Each word is played
out by a stochastic frequency-ranked player, and a uniformly random turn along
that trajectory is sampled. Two knobs control the spread:

* ``exploration_temperature`` perturbs the player's letter ranking, so the
  corpus covers many plausible orderings rather than one canonical one.
* ``uniform_order_probability`` plays a fraction of games in a fully random
  letter order, which reaches unusual states a competent player rarely visits
  but that still occur -- and keeps the model from over-fitting to the habits of
  its bootstrap policy.

Everything is vectorised across the batch: whole games are simulated in closed
form with cumulative sums, so generating millions of states costs seconds.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np


N_LETTERS: int = len(ALPHABET)
_LETTER_A = ord("a")


#: ``letter_targets`` entries at positions that are padding or already revealed.
#: Matches PyTorch's default ``ignore_index`` for cross-entropy.
IGNORE_INDEX: int = -100


@dataclass(frozen=True)
class StateBatch:
    """A batch of encoded training states with their supervision targets.

    Attributes:
        tokens: ``(batch, length)`` board token ids.
        guessed: ``(batch, 26)`` binary already-guessed mask.
        padding: ``(batch, length)`` boolean, ``True`` at padding.
        targets: ``(batch, 26)`` target distribution over hidden letters,
            weighted by occurrence count.
        present: ``(batch, 26)`` binary -- is this letter hidden somewhere?
            This is the quantity a guess is actually ranked by. ``targets``
            answers a subtly different question: in ``banana`` with ``b``
            revealed it assigns ``a`` 2/3 and ``n`` 1/3, because ``a`` occurs
            twice. But guessing ``n`` hits just as surely as guessing ``a`` --
            repetition does not make a letter a better guess. Keeping both lets
            the objective be chosen by measurement rather than assumption.
        letter_targets: ``(batch, length)`` true letter index at each hidden
            position, ``IGNORE_INDEX`` where the position is padding or already
            revealed. This supervises the per-position masked-language-model
            head, which learns orthography directly -- that ``q`` is followed by
            ``u``, that ``_ing`` is a likely ending -- rather than only the
            word-level question of which letters are present somewhere.
    """

    tokens: np.ndarray
    guessed: np.ndarray
    padding: np.ndarray
    targets: np.ndarray
    letter_targets: np.ndarray
    present: np.ndarray

    def __len__(self) -> int:
        return len(self.tokens)


def encode_supervised_states(
    words: Sequence[str],
    boards: Sequence[str],
    guessed_masks: np.ndarray,
    *,
    max_length: int,
) -> StateBatch:
    """Build a supervised batch from explicit (word, board, guessed) triples.

    Used to label states that came from somewhere other than the built-in
    simulator -- self-play, most importantly. The board alone determines which
    positions are hidden, so the targets follow from comparing it to the word.
    """
    batch_size = len(words)
    tokens = np.full((batch_size, max_length), PAD_TOKEN_ID, dtype=np.int64)
    padding = np.ones((batch_size, max_length), dtype=bool)
    letter_targets = np.full((batch_size, max_length), IGNORE_INDEX, dtype=np.int64)
    counts = np.zeros((batch_size, N_LETTERS), dtype=np.float32)

    for row, (word, board) in enumerate(zip(words, boards)):
        padding[row, : len(word)] = False
        for column, (word_char, board_char) in enumerate(zip(word, board)):
            letter = ord(word_char) - _LETTER_A
            if board_char == "_":
                tokens[row, column] = MASK_TOKEN_ID
                if 0 <= letter < N_LETTERS:
                    letter_targets[row, column] = letter
                    counts[row, letter] += 1.0
            else:
                tokens[row, column] = letter + LETTER_TOKEN_OFFSET

    totals = counts.sum(axis=1, keepdims=True).clip(min=1.0)
    return StateBatch(
        tokens=tokens,
        guessed=guessed_masks.astype(np.float32),
        padding=padding,
        targets=counts / totals,
        letter_targets=letter_targets,
        present=(counts > 0).astype(np.float32),
    )


def concatenate_batches(batches: Sequence[StateBatch]) -> StateBatch:
    """Join several batches into one."""
    return StateBatch(
        tokens=np.concatenate([b.tokens for b in batches]),
        guessed=np.concatenate([b.guessed for b in batches]),
        padding=np.concatenate([b.padding for b in batches]),
        targets=np.concatenate([b.targets for b in batches]),
        letter_targets=np.concatenate([b.letter_targets for b in batches]),
        present=np.concatenate([b.present for b in batches]),
    )


def take_rows(batch: StateBatch, indices: np.ndarray) -> StateBatch:
    """Select a subset of rows from a batch."""
    return StateBatch(
        tokens=batch.tokens[indices],
        guessed=batch.guessed[indices],
        padding=batch.padding[indices],
        targets=batch.targets[indices],
        letter_targets=batch.letter_targets[indices],
        present=batch.present[indices],
    )


class GameStateSampler:
    """Generates training states by simulating games over a word list."""

    def __init__(
        self,
        words: list[str],
        *,
        max_length: int,
        exploration_temperature: float = 1.0,
        uniform_order_probability: float = 0.25,
        seed: int = 20260901,
    ) -> None:
        self.max_length = max_length
        self.exploration_temperature = exploration_temperature
        self.uniform_order_probability = uniform_order_probability
        self.rng = np.random.default_rng(seed)

        usable = [w for w in words if 0 < len(w) <= max_length and set(w) <= set(ALPHABET)]
        if not usable:
            raise ValueError("No usable words: check the corpus and max_length.")
        self.words = usable

        self._codes = self._encode_words(usable, max_length)
        self._counts = self._letter_counts(self._codes)
        self._presence = self._counts > 0
        self._n_distinct = self._presence.sum(axis=1)

        # Bootstrap ranking: P(letter appears in a word), in log space.
        presence_rate = self._presence.mean(axis=0).clip(min=1e-6)
        self._log_weights = np.log(presence_rate).astype(np.float32)

    @staticmethod
    def _encode_words(words: list[str], max_length: int) -> np.ndarray:
        """Pack words into a ``(n_words, max_length)`` letter-index matrix, -1 padded."""
        codes = np.full((len(words), max_length), -1, dtype=np.int8)
        for row, word in enumerate(words):
            letters = np.frombuffer(word.encode("ascii"), dtype=np.uint8) - _LETTER_A
            codes[row, : len(word)] = letters
        return codes

    @staticmethod
    def _letter_counts(codes: np.ndarray) -> np.ndarray:
        """Occurrences of each letter per word, ``(n_words, 26)``."""
        counts = np.zeros((len(codes), N_LETTERS), dtype=np.int16)
        for letter in range(N_LETTERS):
            counts[:, letter] = (codes == letter).sum(axis=1)
        return counts

    def sample(self, batch_size: int) -> StateBatch:
        """Draw ``batch_size`` states from freshly simulated games."""
        indices = self.rng.integers(0, len(self.words), size=batch_size)
        return self.sample_for_words(indices)

    def sample_for_words(self, indices: np.ndarray) -> StateBatch:
        """Simulate one game per word index and sample one state from each."""
        presence = self._presence[indices]
        counts = self._counts[indices].astype(np.float32)
        codes = self._codes[indices]
        n_distinct = self._n_distinct[indices]

        order = self._sample_guess_orders(len(indices))
        turn = self._sample_turn(order, presence, n_distinct)

        # rank[b, letter] = the turn at which that letter would be guessed.
        rank = np.argsort(order, axis=1)
        guessed = rank < turn[:, None]

        revealed = guessed & presence
        hidden = presence & ~guessed

        padding = codes < 0
        safe_codes = np.where(padding, 0, codes).astype(np.int64)
        position_revealed = np.take_along_axis(revealed, safe_codes, axis=1)

        tokens = self._build_boards(safe_codes, padding, position_revealed)

        hidden_counts = counts * hidden
        totals = hidden_counts.sum(axis=1, keepdims=True).clip(min=1.0)
        targets = hidden_counts / totals

        # Supervise only the blanks: revealed positions and padding are ignored.
        letter_targets = np.where(
            padding | position_revealed, IGNORE_INDEX, safe_codes
        ).astype(np.int64)

        return StateBatch(
            tokens=tokens,
            guessed=guessed.astype(np.float32),
            padding=padding,
            targets=targets.astype(np.float32),
            letter_targets=letter_targets,
            present=hidden.astype(np.float32),
        )

    def _sample_guess_orders(self, batch_size: int) -> np.ndarray:
        """Sample a guess order per game via the Gumbel-top-k trick.

        Adding Gumbel noise to log-weights and sorting draws an ordering from the
        Plackett-Luce distribution -- sampling letters without replacement in
        proportion to their weight. That is exactly a frequency-ranked player who
        is sometimes wrong about the ranking.
        """
        weights = np.broadcast_to(self._log_weights, (batch_size, N_LETTERS)).copy()

        # A fraction of games use a flat ranking, giving a uniformly random order.
        uniform_rows = self.rng.random(batch_size) < self.uniform_order_probability
        weights[uniform_rows] = 0.0

        uniforms = self.rng.random((batch_size, N_LETTERS)).clip(1e-9, 1 - 1e-9)
        gumbel = -np.log(-np.log(uniforms))
        scores = weights + self.exploration_temperature * gumbel
        return np.argsort(-scores, axis=1)

    def _sample_turn(
        self, order: np.ndarray, presence: np.ndarray, n_distinct: np.ndarray
    ) -> np.ndarray:
        """Pick a uniformly random turn from each game's live trajectory.

        The game is simulated in closed form: a guess is a hit when the letter is
        present, and the game ends at whichever comes first -- the sixth miss, or
        the guess that reveals the final letter. Sampling is inclusive of the
        terminal turn's *pre-guess* state, which is always still winnable and is
        where the hardest decisions live.
        """
        batch_size = len(order)
        rows = np.arange(batch_size)[:, None]
        hits = presence[rows, order]

        hits_cumulative = np.cumsum(hits, axis=1)
        misses_cumulative = np.cumsum(~hits, axis=1)

        solved_turn = np.argmax(hits_cumulative >= n_distinct[:, None], axis=1)

        out_of_lives = misses_cumulative >= MAX_WRONG_GUESSES
        dead_turn = np.where(
            out_of_lives.any(axis=1),
            np.argmax(out_of_lives, axis=1),
            N_LETTERS - 1,
        )

        last_turn = np.minimum(solved_turn, dead_turn)
        return self.rng.integers(0, last_turn + 1)

    @staticmethod
    def _build_boards(
        safe_codes: np.ndarray, padding: np.ndarray, position_revealed: np.ndarray
    ) -> np.ndarray:
        """Render board token ids from letter indices and per-position visibility."""
        tokens = np.full(safe_codes.shape, MASK_TOKEN_ID, dtype=np.int64)
        np.putmask(tokens, position_revealed, safe_codes + LETTER_TOKEN_OFFSET)
        tokens[padding] = PAD_TOKEN_ID
        return tokens


## 5. Model

An encoder-only Transformer with two heads.

* The **presence** head pools the board and asks *which letter is somewhere in this word* -- the decision actually being made.
* The **position** head is a character-level masked language model: for each blank, *what letter fills this slot*. This is the denser signal (one gradient per hidden character rather than one per board) and it is what teaches English orthography -- that `q` takes `u`, that `_ing` is a likely ending. Those cues are precisely what transfers to words the model has never seen.

Per-blank distributions are combined into a word-level score by treating the blanks as independent: `P(absent everywhere) = prod_i (1 - p_i)`. Summing logs avoids underflow, and the result is monotonic in `P(present)`, which is all a ranking needs.


In [ ]:
"""Transformer encoder that predicts which letter is hidden behind the board.

The task is masked language modelling over characters, specialised to Hangman.
Given a partially revealed spelling the model scores all 26 letters by how
likely they are to occupy the remaining blanks, and the solver guesses the
highest-scoring letter it has not already tried.

Two design points do the heavy lifting:

* **Guessed-letter conditioning.** The 26-dim guessed vector is projected into
  the model width and added at every position, so self-attention can reason
  about eliminated letters at each character rather than only at the output. A
  board of ``_ a _ _`` means something very different after five failed guesses
  than it does on turn one.
* **Masked pooling.** Mean and max pooling are concatenated over real positions
  only. Mean captures the overall shape of the word, max captures the single
  most diagnostic position -- a distinctive suffix, say -- which mean pooling
  would otherwise dilute on long words.
"""

from __future__ import annotations

from dataclasses import dataclass

import torch
from torch import Tensor, nn



def _large_negative(tensor: Tensor) -> float:
    """The most negative finite value representable in ``tensor``'s dtype.

    A hard-coded sentinel like ``-1e9`` overflows under mixed precision, where
    activations are float16 and the largest finite magnitude is 65504.
    """
    return torch.finfo(tensor.dtype).min


@dataclass
class ModelConfig:
    """Architecture hyper-parameters.

    Defaults are sized for a 4 GB laptop GPU while staying large enough to model
    English orthography: roughly 3.2M parameters.
    """

    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    dim_feedforward: int = 768
    dropout: float = 0.1
    max_length: int = MAX_WORD_LENGTH
    n_letters: int = len(ALPHABET)
    use_position_head: bool = True


class HangmanTransformer(nn.Module):
    """Encoder-only Transformer scoring each letter's presence in the blanks."""

    def __init__(self, config: ModelConfig | None = None) -> None:
        super().__init__()
        self.config = config or ModelConfig()
        d_model = self.config.d_model

        self.token_embedding = nn.Embedding(VOCABULARY_SIZE, d_model)
        self.position_embedding = nn.Embedding(self.config.max_length, d_model)
        self.guessed_projection = nn.Linear(self.config.n_letters, d_model)
        self.input_norm = nn.LayerNorm(d_model)
        self.input_dropout = nn.Dropout(self.config.dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=self.config.n_heads,
            dim_feedforward=self.config.dim_feedforward,
            dropout=self.config.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=self.config.n_layers,
            norm=nn.LayerNorm(d_model),
            # Nested tensors are unsupported alongside norm_first=True; naming it
            # explicitly keeps PyTorch from warning on every construction.
            enable_nested_tensor=False,
        )

        # Pooled sequence (mean + max) alongside the raw guessed vector.
        head_input = 2 * d_model + self.config.n_letters
        self.head = nn.Sequential(
            nn.LayerNorm(head_input),
            nn.Linear(head_input, d_model),
            nn.GELU(),
            nn.Dropout(self.config.dropout),
            nn.Linear(d_model, self.config.n_letters),
        )

        # Per-position masked-language-model head: what letter sits in this blank?
        self.position_head = (
            nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model),
                nn.GELU(),
                nn.Linear(d_model, self.config.n_letters),
            )
            if self.config.use_position_head
            else None
        )

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.trunc_normal_(module.weight, std=0.02)

    def forward(
        self, tokens: Tensor, guessed: Tensor, padding: Tensor
    ) -> dict[str, Tensor]:
        """Score every letter for a batch of board states.

        Args:
            tokens: ``(batch, length)`` board token ids.
            guessed: ``(batch, 26)`` binary already-guessed mask.
            padding: ``(batch, length)`` boolean, ``True`` at padding positions.

        Returns:
            Dict with ``presence`` ``(batch, 26)`` word-level logits and, when the
            position head is enabled, ``position`` ``(batch, length, 26)``
            per-blank logits. Both are raw, before already-guessed letters are
            masked out.
        """
        batch, length = tokens.shape
        positions = torch.arange(length, device=tokens.device).unsqueeze(0)

        hidden = self.token_embedding(tokens) + self.position_embedding(positions)
        # Broadcast the guessed-letter context to every character position.
        hidden = hidden + self.guessed_projection(guessed).unsqueeze(1)
        hidden = self.input_dropout(self.input_norm(hidden))

        encoded = self.encoder(hidden, src_key_padding_mask=padding)

        real = (~padding).unsqueeze(-1).to(encoded.dtype)
        summed = (encoded * real).sum(dim=1)
        counts = real.sum(dim=1).clamp(min=1.0)
        mean_pooled = summed / counts
        max_pooled = (
            encoded.masked_fill(padding.unsqueeze(-1), _large_negative(encoded))
            .max(dim=1)
            .values
        )

        pooled = torch.cat([mean_pooled, max_pooled, guessed], dim=-1)
        outputs = {"presence": self.head(pooled)}
        if self.position_head is not None:
            outputs["position"] = self.position_head(encoded)
        return outputs

    @staticmethod
    def mask_guessed(logits: Tensor, guessed: Tensor) -> Tensor:
        """Drive already-guessed letters to negligible probability.

        Re-guessing a letter is scored as a strike, so a guessed letter must never
        win the argmax. Applying this at training time too keeps the loss aligned
        with how the model is actually used.
        """
        return logits.masked_fill(guessed > 0, _large_negative(logits))

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def presence_score_from_positions(
    position_logits: Tensor, tokens: Tensor, guessed: Tensor
) -> Tensor:
    """Turn per-blank letter distributions into a word-level presence score.

    Hangman rewards guessing a letter that appears *anywhere* in the word, so the
    per-position distributions must be combined across blanks. Treating the
    blanks as independent,

        P(letter absent everywhere) = prod_i (1 - p_i(letter))

    and the returned score is the negative log of that product. It is monotonic
    in P(letter present), which is all a ranking needs, and summing logs avoids
    the underflow that multiplying many small probabilities would cause.

    Independence is an approximation -- the blanks in a real word are strongly
    correlated -- but it is a well-behaved one, and the pooled presence head is
    trained jointly to capture what it misses.
    """
    blanks = tokens == MASK_TOKEN_ID
    masked = position_logits.masked_fill(
        guessed.unsqueeze(1) > 0, _large_negative(position_logits)
    )
    probabilities = torch.softmax(masked.float(), dim=-1)

    log_absent = torch.log1p(-probabilities.clamp(max=1.0 - 1e-6))
    log_absent = log_absent * blanks.unsqueeze(-1).to(log_absent.dtype)
    return -log_absent.sum(dim=1)


## 6. Guessing policy

Observations arrive one batch per turn, so a single forward pass serves every game still in progress -- necessary to play 250,000 words in reasonable time.

The two heads produce scores on different scales, so `position_weight` is selected on held-out win rate rather than assumed.


In [ ]:
"""The neural guessing policy that drives the game engine."""

from __future__ import annotations

from typing import Sequence

import torch



class NeuralPolicy:
    """Guesses the highest-scoring unguessed letter for each live game.

    Observations arrive one batch per turn, so a single forward pass serves every
    game still in progress. Large batches are split into chunks to bound peak
    memory when playing all 250,000 test words at once.

    The two heads answer the same question differently. The presence head asks
    "which letter is in this word", pooling the whole board. The position head
    asks "what letter fills this specific blank", then those are combined across
    blanks. ``position_weight`` blends their log-scores; it is a validation
    choice, not a guess, and 0.0 or 1.0 recovers either head alone.
    """

    def __init__(
        self,
        model: HangmanTransformer,
        *,
        device: torch.device | str = "cpu",
        chunk_size: int = 2048,
        max_length: int = MAX_WORD_LENGTH,
        position_weight: float = 0.5,
        use_amp: bool | None = None,
    ) -> None:
        self.model = model.to(device).eval()
        self.device = torch.device(device)
        # Chunk size trades Python overhead against peak activation memory. It
        # must stay well under free VRAM: at d_model 384 a single 8192-row
        # activation is ~400 MB in fp32, and when that does not fit alongside a
        # training job the allocator thrashes and inference slows by ~30x.
        self.chunk_size = chunk_size
        self.max_length = max_length
        # Half precision halves activation memory and is ample for ranking 26
        # letters; scores are accumulated in fp32 regardless.
        self.use_amp = (
            self.device.type == "cuda" if use_amp is None else use_amp
        )
        if not 0.0 <= position_weight <= 1.0:
            raise ValueError(f"position_weight must be in [0, 1], got {position_weight}.")
        self.position_weight = position_weight

    @torch.inference_mode()
    def next_guesses(self, observations: Sequence[Observation]) -> list[str]:
        guesses: list[str] = []
        for start in range(0, len(observations), self.chunk_size):
            chunk = observations[start : start + self.chunk_size]
            scores = self.score(chunk)
            guesses.extend(ALPHABET[index] for index in scores.argmax(dim=-1).tolist())
        return guesses

    @torch.inference_mode()
    def score(self, observations: Sequence[Observation]) -> torch.Tensor:
        """Return ``(batch, 26)`` log-scores over letters, guessed ones excluded.

        Exposed separately from :meth:`next_guesses` so several policies can be
        combined at the score level rather than by voting on their final picks --
        averaging distributions keeps the confidence information that a vote
        throws away.
        """
        batch = encode_observations(
            observations, max_length=self.max_length, device=self.device
        )
        tokens, guessed, padding = batch["tokens"], batch["guessed"], batch["padding"]
        with torch.amp.autocast("cuda", enabled=self.use_amp):
            outputs = self.model(tokens, guessed, padding)

        # Already-guessed letters would score as strikes, so they can never win.
        scores = torch.log_softmax(
            self.model.mask_guessed(outputs["presence"].float(), guessed), dim=-1
        )

        if "position" in outputs and self.position_weight > 0.0:
            position_score = presence_score_from_positions(
                outputs["position"], tokens, guessed
            )
            position_scores = torch.log_softmax(
                self.model.mask_guessed(position_score, guessed), dim=-1
            )
            scores = (
                self.position_weight * position_scores
                + (1.0 - self.position_weight) * scores
            )

        return scores


class EnsemblePolicy:
    """Averages the letter distributions of several independently trained models.

    Models trained with different architectures and seeds make different
    mistakes, so averaging their log-scores cancels some of the individual
    error. Combining at the score level rather than by majority vote preserves
    how confident each model was.

    This is the one place where more compute buys score directly, so it is kept
    explicit and optional rather than folded into the single-model path.
    """

    def __init__(self, policies: Sequence[NeuralPolicy], weights: Sequence[float] | None = None) -> None:
        if not policies:
            raise ValueError("An ensemble needs at least one policy.")
        if weights is not None and len(weights) != len(policies):
            raise ValueError(
                f"Got {len(weights)} weights for {len(policies)} policies."
            )
        self.policies = list(policies)
        raw = list(weights) if weights is not None else [1.0] * len(policies)
        total = float(sum(raw))
        if total <= 0.0:
            raise ValueError("Ensemble weights must sum to a positive value.")
        self.weights = [w / total for w in raw]
        self.chunk_size = min(policy.chunk_size for policy in self.policies)

    @torch.inference_mode()
    def next_guesses(self, observations: Sequence[Observation]) -> list[str]:
        guesses: list[str] = []
        for start in range(0, len(observations), self.chunk_size):
            chunk = observations[start : start + self.chunk_size]
            combined = None
            for policy, weight in zip(self.policies, self.weights):
                scores = policy.score(chunk) * weight
                combined = scores if combined is None else combined + scores
            guesses.extend(
                ALPHABET[index] for index in combined.argmax(dim=-1).tolist()
            )
        return guesses


## 7. Self-play (DAgger)

The bootstrap sampler imitates a frequency-ranked player, but the trained model reveals letters in a different order and therefore visits a different part of the state space. Fitting on one distribution while acting in another is textbook covariate shift.

DAgger addresses it directly: let the current policy act, then train on the states it actually reaches. The model plays real training words, every intermediate board is labelled with the answer, and those states are mixed back in. The model still only ever sees an `Observation` while playing -- the word is used solely to build the label afterwards.

The simulated sampler is deliberately kept in the mix. Training purely on self-play would let the model revisit only its own habits, and states it has learned to avoid would vanish from the training set.


In [ ]:
"""Self-play state collection (DAgger).

The bootstrap sampler in :mod:`hangman.dataset` simulates games with a
stochastic frequency-ranked player. That is a reasonable stand-in, but it is not
the player whose states actually matter -- the trained model reveals letters in
a different order, so it visits a different part of the state space. Fitting on
one distribution and acting in another is exactly the covariate shift that
DAgger addresses: let the current policy act, then train on the states it
actually reaches.

Concretely, the model plays real games on *training* words, every intermediate
board is labelled with the answer, and those states are mixed back into the
training stream. The model still only ever sees an ``Observation`` while
playing; the word is used solely to construct the label afterwards, exactly as
in ordinary supervised learning.
"""

from __future__ import annotations

import numpy as np
import torch


N_LETTERS = len(ALPHABET)


@torch.inference_mode()
def collect_self_play_states(
    model: HangmanTransformer,
    words: list[str],
    *,
    device: torch.device | str,
    max_length: int,
    position_weight: float = 0.5,
    chunk_size: int = 8192,
) -> StateBatch:
    """Play ``words`` with ``model`` and label every state along the way.

    Returns one labelled state per turn per game. Longer games contribute more
    states, which is the desired weighting: those are the words the model finds
    hard, and they are where the remaining wins are.
    """
    was_training = model.training
    model.eval()
    policy = NeuralPolicy(
        model, device=device, chunk_size=chunk_size, position_weight=position_weight
    )

    states = [GameState(word=word) for word in words]
    active = [state for state in states if not state.is_over]

    recorded_words: list[str] = []
    recorded_boards: list[str] = []
    recorded_guessed: list[np.ndarray] = []

    while active:
        observations = [state.observation for state in active]

        for state, observation in zip(active, observations):
            mask = np.zeros(N_LETTERS, dtype=np.float32)
            for letter in observation.guessed_letters:
                index = ALPHABET.find(letter)
                if index >= 0:
                    mask[index] = 1.0
            recorded_words.append(state.word)
            recorded_boards.append(observation.board)
            recorded_guessed.append(mask)

        for state, guess in zip(active, policy.next_guesses(observations)):
            state.apply_guess(guess)
        active = [state for state in active if not state.is_over]

    if was_training:
        model.train()

    return encode_supervised_states(
        recorded_words,
        recorded_boards,
        np.stack(recorded_guessed),
        max_length=max_length,
    )


class MixedStateSampler:
    """Draws each batch from simulated states and self-play states in a fixed ratio.

    Training purely on self-play states would be a mistake: the model would only
    ever revisit its own habits, and states it has learned to avoid -- including
    the ones it reaches after an unlucky opening -- would vanish from the
    training set. Keeping the simulated sampler in the mix preserves that
    coverage while the self-play share corrects the distribution shift.
    """

    def __init__(
        self,
        base: GameStateSampler,
        *,
        self_play_fraction: float = 0.5,
        seed: int = 20260901,
    ) -> None:
        if not 0.0 <= self_play_fraction <= 1.0:
            raise ValueError(
                f"self_play_fraction must be in [0, 1], got {self_play_fraction}."
            )
        self.base = base
        self.self_play_fraction = self_play_fraction
        self.rng = np.random.default_rng(seed)
        self._buffer: StateBatch | None = None

    @property
    def buffer_size(self) -> int:
        return 0 if self._buffer is None else len(self._buffer)

    def refresh(self, batch: StateBatch) -> None:
        """Replace the self-play buffer with freshly collected states."""
        self._buffer = batch

    def sample(self, batch_size: int) -> StateBatch:
        if self._buffer is None or self.self_play_fraction <= 0.0:
            return self.base.sample(batch_size)

        n_self_play = min(
            int(round(batch_size * self.self_play_fraction)), len(self._buffer)
        )
        n_simulated = batch_size - n_self_play
        if n_self_play == 0:
            return self.base.sample(batch_size)

        indices = self.rng.integers(0, len(self._buffer), size=n_self_play)
        parts = [take_rows(self._buffer, indices)]
        if n_simulated > 0:
            parts.insert(0, self.base.sample(n_simulated))
        return concatenate_batches(parts)


## 8. Weight averaging

A decaying learning rate still leaves the final weights rattling around the minimum rather than sitting in it. An exponential moving average of the trajectory lands nearer the centre of the basin and usually generalises a little better -- for one extra copy of the weights and no extra inference cost.

This is averaging along a single run, not an ensemble of separate models: it stays one model at inference time.


In [ ]:
"""Exponential moving average of model weights.

SGD with a decaying learning rate still leaves the final weights rattling around
the minimum rather than sitting in it. An exponential moving average of the
trajectory lands closer to the centre of the basin, which usually generalises
slightly better than any single iterate -- and unlike an ensemble it costs one
extra copy of the weights and no extra inference.

That distinction matters here: averaging along a single training run is a
better-conditioned estimate of the same model, not a bag of separate models
stacked to buy score. It stays one model at inference time.
"""

from __future__ import annotations

from copy import deepcopy

import torch
from torch import nn


class WeightAverager:
    """Maintains an exponential moving average of a model's parameters.

    Args:
        model: The model whose weights are tracked.
        decay: EMA decay. Higher averages over a longer window; 0.999 keeps an
            effective horizon of roughly the last thousand steps.
        warmup_steps: Early weights change fast and are not worth averaging, so
            the average simply tracks the model until this many updates have
            happened.
    """

    def __init__(
        self, model: nn.Module, *, decay: float = 0.999, warmup_steps: int = 1_000
    ) -> None:
        if not 0.0 < decay < 1.0:
            raise ValueError(f"decay must be in (0, 1), got {decay}.")
        self.decay = decay
        self.warmup_steps = warmup_steps
        self.updates = 0
        self.shadow = deepcopy(model).eval()
        for parameter in self.shadow.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        """Fold the current weights into the average."""
        self.updates += 1
        # Ramp the decay in, so the average is not dragged by its initialisation.
        decay = 0.0 if self.updates <= self.warmup_steps else self.decay

        for shadow_param, model_param in zip(
            self.shadow.parameters(), model.parameters()
        ):
            shadow_param.mul_(decay).add_(model_param.detach(), alpha=1.0 - decay)

        # Buffers (LayerNorm has none here, but be safe) are copied, not averaged.
        for shadow_buffer, model_buffer in zip(
            self.shadow.buffers(), model.buffers()
        ):
            shadow_buffer.copy_(model_buffer)

    @property
    def model(self) -> nn.Module:
        """The averaged model, ready for evaluation."""
        return self.shadow


## 9. Training

The loss combines the word-level presence objective with the per-blank language-model objective.

Model selection does **not** use the loss. The competition scores games won, not per-turn letter accuracy, and the two diverge: loss plateaus early while win rate keeps climbing. So evaluation runs the real 6-life engine on held-out words and checkpoints on win rate.


In [ ]:
"""Training loop for the Hangman transformer.

The loss is a cross-entropy against the distribution of letters still hidden,
with already-guessed letters masked out of the softmax exactly as they are at
inference time. Optimising that distribution is a proxy, though -- the
competition scores *games won*, not per-turn letter accuracy. So model selection
runs the real 6-life engine on held-out words and checkpoints on win rate. The
two can diverge: a model that shaves loss by sharpening already-easy mid-game
states while mishandling the opening turn will win fewer games, and only the
game-level metric notices.
"""

from __future__ import annotations

import json
import math
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from torch import Tensor, nn



@dataclass
class TrainingConfig:
    """Optimisation and evaluation settings."""

    steps: int = 12_000
    batch_size: int = 512
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    warmup_steps: int = 500
    grad_clip: float = 1.0
    eval_every: int = 1_000
    eval_words: int = 2_000
    log_every: int = 100
    uniform_order_probability: float = 0.25
    exploration_temperature: float = 1.0
    seed: int = 20260901
    amp: bool = True
    #: Weight on the per-blank language-model loss, relative to the presence loss.
    position_loss_weight: float = 1.0
    #: Blend of the two heads at inference; selected on held-out win rate.
    position_weight: float = 0.5

    # --- DAgger self-play ---
    #: Step at which to start mixing in states from the model's own play. Kept
    #: off early: a barely-trained model's trajectories are not worth imitating.
    self_play_start_step: int = 0
    #: How often to re-collect self-play states, in steps.
    self_play_refresh_every: int = 4_000
    #: Training words to play when refreshing the buffer.
    self_play_words: int = 20_000
    #: Share of each batch drawn from the self-play buffer.
    self_play_fraction: float = 0.5

    #: "count" weights letters by how often they repeat; "binary" scores only
    #: whether a letter is present, which is what a guess is ranked by.
    presence_objective: str = "count"
    #: EMA decay for weight averaging. 0 disables.
    ema_decay: float = 0.0
    ema_warmup_steps: int = 1_000


def cross_entropy_over_hidden_letters(
    logits: Tensor, targets: Tensor, guessed: Tensor
) -> Tensor:
    """Cross-entropy between the model's letter distribution and the hidden one.

    Guessed letters are removed from the softmax so the model never spends
    probability mass on a letter it is not allowed to pick.
    """
    masked = HangmanTransformer.mask_guessed(logits, guessed)
    log_probabilities = torch.log_softmax(masked.float(), dim=-1)
    return -(targets * log_probabilities).sum(dim=-1).mean()


def masked_language_model_loss(
    position_logits: Tensor, letter_targets: Tensor
) -> Tensor:
    """Cross-entropy on each blank's true letter, ignoring revealed positions."""
    return nn.functional.cross_entropy(
        position_logits.float().flatten(0, 1),
        letter_targets.flatten(),
        ignore_index=IGNORE_INDEX,
    )


def binary_presence_loss(
    logits: Tensor, present: Tensor, guessed: Tensor
) -> Tensor:
    """Per-letter binary cross-entropy on "is this letter hidden somewhere?".

    This matches what the policy actually ranks by. The count-weighted
    cross-entropy alternative implicitly rewards letters that repeat, which is
    not what makes a guess good -- guessing ``n`` in ``banana`` hits exactly as
    reliably as guessing ``a``.

    Already-guessed letters are excluded from the loss: their answer is known,
    so scoring them would only teach the model something it is never asked.
    """
    per_letter = nn.functional.binary_cross_entropy_with_logits(
        logits.float(), present, reduction="none"
    )
    unguessed = 1.0 - guessed
    return (per_letter * unguessed).sum() / unguessed.sum().clamp(min=1.0)


def combined_loss(
    outputs: dict[str, Tensor],
    batch: dict[str, Tensor],
    position_weight: float,
    presence_objective: str = "count",
) -> tuple[Tensor, dict[str, float]]:
    """Word-level presence loss plus the per-blank language-model loss.

    The two are complementary. Presence supervises the decision actually being
    made -- which letter to call next. The blank-level objective is a denser
    signal: one gradient per hidden character rather than one per board, which
    is what teaches the model English spelling well enough to generalise to
    words it has never seen.
    """
    if presence_objective == "binary":
        presence = binary_presence_loss(
            outputs["presence"], batch["present"], batch["guessed"]
        )
    elif presence_objective == "count":
        presence = cross_entropy_over_hidden_letters(
            outputs["presence"], batch["targets"], batch["guessed"]
        )
    else:
        raise ValueError(
            f"presence_objective must be 'count' or 'binary', got {presence_objective!r}."
        )
    parts = {"presence": presence.item()}

    total = presence
    if "position" in outputs and position_weight > 0.0:
        position = masked_language_model_loss(
            outputs["position"], batch["letter_targets"]
        )
        total = total + position_weight * position
        parts["position"] = position.item()

    return total, parts


def to_tensors(batch: StateBatch, device: torch.device) -> dict[str, Tensor]:
    return {
        "tokens": torch.from_numpy(batch.tokens).to(device, non_blocking=True),
        "guessed": torch.from_numpy(batch.guessed).to(device, non_blocking=True),
        "padding": torch.from_numpy(batch.padding).to(device, non_blocking=True),
        "targets": torch.from_numpy(batch.targets).to(device, non_blocking=True),
        "letter_targets": torch.from_numpy(batch.letter_targets).to(
            device, non_blocking=True
        ),
        "present": torch.from_numpy(batch.present).to(device, non_blocking=True),
    }


def learning_rate_at(step: int, config: TrainingConfig) -> float:
    """Linear warmup into a cosine decay."""
    if step < config.warmup_steps:
        return config.learning_rate * (step + 1) / config.warmup_steps
    progress = (step - config.warmup_steps) / max(1, config.steps - config.warmup_steps)
    return config.learning_rate * 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))


@torch.no_grad()
def evaluate_win_rate(
    model: HangmanTransformer,
    words: list[str],
    device: torch.device,
    *,
    position_weight: float = 0.5,
) -> dict[str, float]:
    """Play real games on held-out words and report the competition metrics."""
    was_training = model.training
    model.eval()
    policy = NeuralPolicy(model, device=device, position_weight=position_weight)
    metrics = summarise(play_games(words, policy))
    if was_training:
        model.train()
    return metrics


def train(
    train_words: list[str],
    validation_words: list[str],
    *,
    model_config: ModelConfig | None = None,
    training_config: TrainingConfig | None = None,
    output_dir: Path = Path("artifacts"),
    device: torch.device | str | None = None,
) -> tuple[HangmanTransformer, dict[str, object]]:
    """Fit the model, checkpointing whenever held-out win rate improves."""
    model_config = model_config or ModelConfig()
    config = training_config or TrainingConfig()
    device = torch.device(
        device or ("cuda" if torch.cuda.is_available() else "cpu")
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(config.seed)
    np.random.seed(config.seed)

    model = HangmanTransformer(model_config).to(device)
    base_sampler = GameStateSampler(
        train_words,
        max_length=model_config.max_length,
        exploration_temperature=config.exploration_temperature,
        uniform_order_probability=config.uniform_order_probability,
        seed=config.seed,
    )
    self_play_enabled = (
        config.self_play_start_step > 0 and config.self_play_fraction > 0.0
    )
    sampler = MixedStateSampler(
        base_sampler,
        self_play_fraction=config.self_play_fraction if self_play_enabled else 0.0,
        seed=config.seed,
    )
    self_play_rng = np.random.default_rng(config.seed)
    averager = (
        WeightAverager(
            model, decay=config.ema_decay, warmup_steps=config.ema_warmup_steps
        )
        if config.ema_decay > 0.0
        else None
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.98),
    )
    use_amp = config.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    eval_words = validation_words[: config.eval_words]
    history: list[dict[str, float]] = []
    best_win_rate = -1.0
    best_path = output_dir / "best_model.pt"

    print(f"device={device}  parameters={model.count_parameters():,}")
    print(f"train words={len(train_words):,}  eval words={len(eval_words):,}")
    print(f"steps={config.steps:,}  batch={config.batch_size}\n")

    model.train()
    running_loss = 0.0
    running_parts: dict[str, float] = {}
    started = time.perf_counter()

    for step in range(config.steps):
        if self_play_enabled and step >= config.self_play_start_step:
            steps_in = step - config.self_play_start_step
            if steps_in % config.self_play_refresh_every == 0:
                chosen = self_play_rng.choice(
                    len(train_words),
                    size=min(config.self_play_words, len(train_words)),
                    replace=False,
                )
                collected = collect_self_play_states(
                    model,
                    [train_words[i] for i in chosen],
                    device=device,
                    max_length=model_config.max_length,
                    position_weight=config.position_weight,
                )
                sampler.refresh(collected)
                print(
                    f"  self-play @ {step:,}: collected {len(collected):,} states "
                    f"from {len(chosen):,} games"
                )

        for group in optimizer.param_groups:
            group["lr"] = learning_rate_at(step, config)

        batch = to_tensors(sampler.sample(config.batch_size), device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs = model(batch["tokens"], batch["guessed"], batch["padding"])
            loss, loss_parts = combined_loss(
                outputs,
                batch,
                config.position_loss_weight,
                presence_objective=config.presence_objective,
            )

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        if averager is not None:
            averager.update(model)

        running_loss += loss.item()
        running_parts = {
            key: running_parts.get(key, 0.0) + value
            for key, value in loss_parts.items()
        }

        if (step + 1) % config.log_every == 0:
            elapsed = time.perf_counter() - started
            mean_loss = running_loss / config.log_every
            rate = (step + 1) / elapsed
            breakdown = "  ".join(
                f"{key} {value / config.log_every:.4f}"
                for key, value in running_parts.items()
            )
            print(
                f"step {step + 1:>6,}/{config.steps:,}  loss {mean_loss:.4f}  "
                f"({breakdown})  lr {learning_rate_at(step, config):.2e}  "
                f"{rate:.1f} steps/s"
            )
            running_loss = 0.0
            running_parts = {}

        is_last = step + 1 == config.steps
        if (step + 1) % config.eval_every == 0 or is_last:
            # Once weight averaging is active it is what will be shipped, so it
            # is what gets measured and checkpointed.
            candidate = averager.model if averager is not None else model
            metrics = evaluate_win_rate(
                candidate, eval_words, device, position_weight=config.position_weight
            )
            metrics["step"] = step + 1
            history.append(metrics)
            marker = ""
            if metrics["win_rate"] > best_win_rate:
                best_win_rate = metrics["win_rate"]
                torch.save(
                    {
                        "model_state": candidate.state_dict(),
                        "model_config": asdict(model_config),
                        "win_rate": best_win_rate,
                        "step": step + 1,
                    },
                    best_path,
                )
                marker = "  <- best, checkpointed"
            print(
                f"  eval @ {step + 1:,}: win rate {metrics['win_rate']:.2f}%  "
                f"mean wrong {metrics['mean_wrong']:.3f}{marker}"
            )

    summary = {
        "best_win_rate": best_win_rate,
        "checkpoint": str(best_path),
        "history": history,
        "model_config": asdict(model_config),
        "training_config": asdict(config),
        "minutes": (time.perf_counter() - started) / 60.0,
    }
    (output_dir / "training_summary.json").write_text(json.dumps(summary, indent=2))

    # Restore the best checkpoint rather than returning the final-step weights.
    model.load_state_dict(torch.load(best_path, map_location=device)["model_state"])
    return model, summary


def load_model(
    checkpoint_path: str | Path, device: torch.device | str = "cpu"
) -> HangmanTransformer:
    """Rebuild a model from a saved checkpoint, config included.

    Checkpoints written before the per-position head existed have no
    ``use_position_head`` entry in their config. Rather than defaulting it and
    failing on a shape mismatch, the flag is recovered from the saved weights
    themselves, so older runs stay loadable for ablation comparisons.
    """
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state = checkpoint["model_state"]
    config_fields = dict(checkpoint["model_config"])
    config_fields.setdefault(
        "use_position_head",
        any(key.startswith("position_head.") for key in state),
    )

    model = HangmanTransformer(ModelConfig(**config_fields))
    model.load_state_dict(state)
    return model.to(device).eval()


## 10. Reference baselines

Included to establish the bar the learned model has to clear, and to show *why* a statistical approach is not enough here.


In [ ]:
"""Non-neural reference policies.

These exist to establish the bar, not to be submitted. The competition weights
learned models above "simple N-Gram or statistical frequency baselines", so the
neural solver has to beat these convincingly for the approach to be justified.

Two baselines, in increasing order of strength:

* :class:`StaticFrequencyPolicy` -- a fixed letter order, ignoring the board.
  This is what ``sample_submission.csv`` does.
* :class:`PatternMatchingPolicy` -- filter the training vocabulary down to the
  words consistent with the board, then guess the letter appearing in the most
  survivors. Strong on paper, but it can only ever recognise spellings it has
  already seen, and train/test here are disjoint.
"""

from __future__ import annotations

from collections import Counter
from typing import Sequence

import numpy as np


# Letters ordered by P(letter appears in a word), measured on train.txt.
# This differs from raw token frequency -- what matters in Hangman is whether a
# letter is present at all, not how often it repeats.
TRAIN_PRESENCE_ORDER: str = "eiarnostlcudpmhgybfvkwzxqj"


def presence_order(words: Sequence[str]) -> str:
    """Rank letters by the fraction of ``words`` containing them at least once."""
    counts = Counter()
    for word in words:
        counts.update(set(word))
    ranked = sorted(ALPHABET, key=lambda letter: -counts[letter])
    return "".join(ranked)


class StaticFrequencyPolicy:
    """Guess a fixed letter order, skipping letters already guessed.

    Ignores the board entirely, so it is the weakest sensible reference point.
    """

    def __init__(self, order: str = TRAIN_PRESENCE_ORDER) -> None:
        self.order = order

    def next_guesses(self, observations: Sequence[Observation]) -> Sequence[str]:
        return [self._guess(obs) for obs in observations]

    def _guess(self, observation: Observation) -> str:
        guessed = observation.guessed_letters
        for letter in self.order:
            if letter not in guessed:
                return letter
        return self.order[-1]


class PatternMatchingPolicy:
    """Constrain the training vocabulary to the board, then vote on letters.

    A word is consistent with the board when, for every letter already guessed,
    its occurrence positions exactly match the board's. That single condition
    covers hits and misses at once: a correct guess reveals *all* of its
    occurrences, so a masked position can never hold an already-guessed letter.

    Words are bucketed by length and stored as integer matrices so each turn is
    a handful of vectorised comparisons rather than a Python loop over the
    vocabulary.
    """

    def __init__(self, words: Sequence[str], *, fallback_order: str | None = None) -> None:
        self.fallback_order = fallback_order or presence_order(words)
        self._codes: dict[int, np.ndarray] = {}
        self._presence: dict[int, np.ndarray] = {}

        by_length: dict[int, list[str]] = {}
        for word in words:
            if set(word) <= set(ALPHABET):
                by_length.setdefault(len(word), []).append(word)

        for length, bucket in by_length.items():
            codes = np.frombuffer("".join(bucket).encode("ascii"), dtype=np.uint8)
            codes = codes.reshape(len(bucket), length)
            self._codes[length] = codes
            # presence[i, j] -> does word i contain letter j at least once
            presence = np.zeros((len(bucket), len(ALPHABET)), dtype=bool)
            for index, letter in enumerate(ALPHABET):
                presence[:, index] = (codes == ord(letter)).any(axis=1)
            self._presence[length] = presence

    def next_guesses(self, observations: Sequence[Observation]) -> Sequence[str]:
        return [self._guess(obs) for obs in observations]

    def _guess(self, observation: Observation) -> str:
        guessed = observation.guessed_letters
        candidates = self._consistent_words(observation)

        if candidates is not None and candidates.size:
            presence = self._presence[observation.length][candidates]
            scores = presence.sum(axis=0).astype(np.float64)
            for letter in guessed:
                if letter in ALPHABET:
                    scores[ALPHABET.index(letter)] = -1.0
            best = int(np.argmax(scores))
            if scores[best] > 0:
                return ALPHABET[best]

        # No consistent word survived: fall back to corpus-wide frequency.
        for letter in self.fallback_order:
            if letter not in guessed:
                return letter
        return self.fallback_order[-1]

    def _consistent_words(self, observation: Observation) -> np.ndarray | None:
        """Indices of vocabulary words compatible with the board, or ``None``."""
        codes = self._codes.get(observation.length)
        if codes is None:
            return None

        board = np.frombuffer(observation.board.encode("ascii"), dtype=np.uint8)
        keep = np.ones(len(codes), dtype=bool)
        for letter in observation.guessed_letters:
            if letter not in ALPHABET:
                continue
            board_positions = board == ord(letter)
            word_positions = codes == ord(letter)
            keep &= ~(word_positions != board_positions).any(axis=1)
            if not keep.any():
                return np.empty(0, dtype=np.int64)

        # Masked positions must not hold a letter that is already revealed.
        revealed = board != ord(MASK_CHAR)
        if revealed.any():
            keep &= (codes[:, revealed] == board[revealed]).all(axis=1)

        return np.flatnonzero(keep)


## 11. Submission writing and validation

The written file is re-scored by replaying its guesses through the engine, so the number reported is one an independent simulation produced -- not one the solver claimed about itself.


In [ ]:
"""Writing and validating the competition submission file.

The submission is a flat CSV with a header and one row per test word:

    word_id,guessed_letters_string
    0,etaonsrhld
    ...

``word_id`` is the zero-based line index of the word in ``test.txt``, and
``guessed_letters_string`` is the chronological sequence of guesses the solver
actually made while playing that word.

The validator here re-scores a written file with the same engine the grader
describes, so that the number reported locally is the number the leaderboard
will compute -- not a number the solver claimed for itself.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import pandas as pd


WORD_ID_COLUMN: str = "word_id"
GUESSES_COLUMN: str = "guessed_letters_string"
EXPECTED_ROW_COUNT: int = 250_000

#: Divisor relating mean wrong guesses to the efficiency penalty Kaggle folds
#: into the displayed score. Fitted from a real submission: a replay win rate of
#: 53.5292% with 4.4155 mean wrong guesses scored 53.4766 on the public
#: leaderboard, and 4.4155 / 84 = 0.05257 closes that gap to four decimals.
#:
#: The rules describe the tie-break qualitatively but not its constant, so this
#: is an empirical estimate from a single data point. It affects only the
#: predicted display score, never the win rate that decides rank -- more words
#: solved always outranks fewer.
EFFICIENCY_PENALTY_DIVISOR: float = 84.0


def write_submission(
    results: Sequence[GameResult],
    path: str | Path,
    *,
    expected_rows: int | None = EXPECTED_ROW_COUNT,
) -> Path:
    """Write ``results`` to ``path`` in the required CSV schema.

    Args:
        results: Finished games, in test-set order.
        path: Destination file, conventionally ``submission.csv``.
        expected_rows: Row count to assert before writing. Pass ``None`` to skip
            the check (useful when writing a validation-split submission).

    Raises:
        ValueError: If the row count does not match ``expected_rows``.
    """
    if expected_rows is not None and len(results) != expected_rows:
        raise ValueError(
            f"Expected {expected_rows} results, got {len(results)}. "
            "The submission must have exactly one row per test word."
        )

    frame = pd.DataFrame(
        {
            WORD_ID_COLUMN: range(len(results)),
            GUESSES_COLUMN: [result.guess_string for result in results],
        }
    )
    path = Path(path)
    frame.to_csv(path, index=False)
    return path


@dataclass(frozen=True)
class ValidationReport:
    """Outcome of re-scoring a submission file against the test words."""

    rows: int
    win_rate: float
    total_wrong: int
    mean_wrong: float
    schema_errors: list[str]
    wasted_guesses: int

    @property
    def is_valid(self) -> bool:
        return not self.schema_errors

    @property
    def estimated_leaderboard_score(self) -> float:
        """Win rate less the estimated efficiency tie-break Kaggle displays."""
        return self.win_rate - self.mean_wrong / EFFICIENCY_PENALTY_DIVISOR

    def __str__(self) -> str:
        status = "VALID" if self.is_valid else f"INVALID ({len(self.schema_errors)} errors)"
        lines = [
            f"Submission: {status}",
            f"  rows            : {self.rows:,}",
            f"  win rate        : {self.win_rate:.4f}%",
            f"  total wrong     : {self.total_wrong:,}",
            f"  mean wrong/word : {self.mean_wrong:.4f}",
            f"  wasted guesses  : {self.wasted_guesses:,} (emitted after termination)",
            f"  est. LB score   : {self.estimated_leaderboard_score:.4f}",
        ]
        lines.extend(f"  ! {error}" for error in self.schema_errors[:10])
        if len(self.schema_errors) > 10:
            lines.append(f"  ! ... and {len(self.schema_errors) - 10} more")
        return "\n".join(lines)


def validate_submission(
    path: str | Path,
    test_words: Sequence[str],
    *,
    expected_rows: int | None = EXPECTED_ROW_COUNT,
) -> ValidationReport:
    """Re-score a submission file by replaying its guesses through the engine.

    Every guess is fed to a fresh :class:`GameState` until the game terminates,
    exactly as the grader's 6-strike lockout describes. Guesses appearing after
    termination are counted as ``wasted_guesses`` -- they are ignored by the
    scorer, so they cost nothing, but a large count signals a solver that does
    not know when to stop.
    """
    frame = pd.read_csv(path, dtype={WORD_ID_COLUMN: "Int64", GUESSES_COLUMN: str})
    frame[GUESSES_COLUMN] = frame[GUESSES_COLUMN].fillna("")

    schema_errors: list[str] = []
    for column in (WORD_ID_COLUMN, GUESSES_COLUMN):
        if column not in frame.columns:
            schema_errors.append(f"missing required column {column!r}")
    if schema_errors:
        return ValidationReport(len(frame), 0.0, 0, 0.0, schema_errors, 0)

    if expected_rows is not None and len(frame) != expected_rows:
        schema_errors.append(f"expected {expected_rows} rows, found {len(frame)}")
    if len(frame) != len(test_words):
        schema_errors.append(
            f"row count {len(frame)} does not match {len(test_words)} test words"
        )
    if list(frame[WORD_ID_COLUMN]) != list(range(len(frame))):
        schema_errors.append(f"{WORD_ID_COLUMN} must be 0..{len(frame) - 1} in order")

    allowed = set(ALPHABET)
    results: list[GameResult] = []
    wasted_guesses = 0

    for row_index, (guess_string, word) in enumerate(
        zip(frame[GUESSES_COLUMN], test_words)
    ):
        invalid = sorted(set(guess_string) - allowed)
        if invalid:
            schema_errors.append(
                f"row {row_index}: non a-z characters in guesses: {invalid}"
            )

        state = GameState(word=word)
        consumed = 0
        for guess in guess_string:
            if state.is_over:
                break
            state.apply_guess(guess)
            consumed += 1
        wasted_guesses += len(guess_string) - consumed

        results.append(
            GameResult(
                word=word,
                guess_string=guess_string,
                solved=state.is_solved,
                wrong_guesses=state.wrong_guesses,
            )
        )

    metrics = summarise(results)
    return ValidationReport(
        rows=len(frame),
        win_rate=metrics["win_rate"],
        total_wrong=int(metrics["total_wrong"]),
        mean_wrong=metrics["mean_wrong"],
        schema_errors=schema_errors,
        wasted_guesses=wasted_guesses,
    )


## 12. Data

Load the vocabulary and split off a held-out slice for model selection.


In [ ]:
# Locate the competition data. Kaggle has mounted it under a few different
# paths over time, so try each rather than hard-coding one.
CANDIDATE_DATA_DIRS = [
    Path("/kaggle/input/brand-buzzword-hackathon"),
    Path("/kaggle/input/competitions/brand-buzzword-hackathon"),
    Path("data"),
]
DATA_DIR = next((p for p in CANDIDATE_DATA_DIRS if (p / TRAIN_FILENAME).exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        f"Could not find {TRAIN_FILENAME} in any of: {CANDIDATE_DATA_DIRS}"
    )

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("artifacts")
print(f"data  : {DATA_DIR}")
print(f"output: {OUTPUT_DIR}")

train_vocabulary = load_words(DATA_DIR / TRAIN_FILENAME)
split = split_words(train_vocabulary, validation_size=20_000)
print(split)
print(describe_vocabulary(train_vocabulary))


## 13. Training

Trains from scratch on `train.txt`. Nothing else is read: no external corpora, no pretrained weights, no API calls. Checkpointing is on held-out **win rate**, not loss.


In [ ]:
# Train from scratch on train.txt. Nothing else is read: no external corpora,
# no pretrained weights, no network access.
model, summary = train(
    split.train,
    split.validation,
    model_config=ModelConfig(
        d_model=448,
        n_heads=8,
        n_layers=8,
        dim_feedforward=1344,
    ),
    training_config=TrainingConfig(
        steps=34000,
        batch_size=2048,
        learning_rate=0.00045,
        eval_every=5000,
        eval_words=5000,
        position_weight=0.5,
        self_play_start_step=0,
        self_play_refresh_every=5000,
        self_play_words=20000,
        self_play_fraction=0.5,
    ),
    output_dir=OUTPUT_DIR,
)
print(f"best held-out win rate: {summary['best_win_rate']:.2f}%")


## 14. Playing the test set

Each word is played turn by turn against the real 6-life engine, and the guess sequence the model actually produced is recorded. The finished file is then re-scored by an independent replay.


In [ ]:
POSITION_WEIGHT = 0.5


In [ ]:
# Play every test word and write the submission.
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_words = load_words(DATA_DIR / TEST_FILENAME)
model = model.to(device).eval()
policy = NeuralPolicy(model, device=device, position_weight=POSITION_WEIGHT)

results = play_games(test_words, policy)
metrics = summarise(results)
print(f"win rate    : {metrics['win_rate']:.4f}%")
print(f"mean wrong  : {metrics['mean_wrong']:.4f}")

write_submission(results, "submission.csv")
report = validate_submission("submission.csv", test_words)
print(report)
assert report.is_valid, "submission failed validation"
